# Narration director: QLoRA fine-tune (Colab T4)

In [ ]:
import torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

In [ ]:
!pip -q install -U peft bitsandbytes datasets accelerate

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
BASE = "/content/drive/MyDrive/audiobook_genai"
DATA_DIR = f"{BASE}/splits"

SMOKE = True
MAX_LEN = 768
EPOCHS = 2
LR = 2e-4
BATCH = 4
ACCUM = 4

OUT_DIR = f"{BASE}/run1_smoke" if SMOKE else f"{BASE}/run1"
SAVE_DIR = f"{BASE}/adapter_smoke" if SMOKE else f"{BASE}/adapter_final"

In [ ]:
import json, os
from transformers import AutoTokenizer
from datasets import Dataset

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_rows = read_jsonl(f"{DATA_DIR}/train.jsonl")
val_rows = read_jsonl(f"{DATA_DIR}/val.jsonl")
if SMOKE:
    train_rows, val_rows = train_rows[:160], val_rows[:40]

tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "right"

def encode(row):
    p = tok(row["prompt"], add_special_tokens=False)["input_ids"]
    c = tok(" " + row["completion"], add_special_tokens=False)["input_ids"] + [tok.eos_token_id]
    return {"input_ids": p + c,
            "attention_mask": [1] * (len(p) + len(c)),
            "labels": [-100] * len(p) + c}

def build(rows):
    enc = [encode(r) for r in rows]
    keep = [e for e in enc if len(e["input_ids"]) <= MAX_LEN]
    print(f"kept {len(keep)}/{len(enc)} (dropped {len(enc) - len(keep)} longer than {MAX_LEN} tokens)")
    return Dataset.from_list(keep)

train_ds, val_ds = build(train_rows), build(val_rows)
lens = [len(x) for x in train_ds["input_ids"]]
print("average tokens:", sum(lens) // len(lens), "| longest:", max(lens))

In [ ]:
import transformers
from packaging import version
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)

dtype_key = "dtype" if version.parse(transformers.__version__) >= version.parse("4.56") else "torch_dtype"
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb,
                                             device_map={"": 0}, **{dtype_key: torch.float16})
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                  "gate_proj", "up_proj", "down_proj"])
model = get_peft_model(model, lora)
for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.float()
model.config.use_cache = False
model.print_trainable_parameters()

In [ ]:
import math, inspect
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq, set_seed
from transformers.trainer_utils import get_last_checkpoint

set_seed(42)

total_steps = 10 if SMOKE else math.ceil(len(train_ds) / (BATCH * ACCUM)) * EPOCHS
cfg = dict(
    output_dir=OUT_DIR,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=ACCUM,
    num_train_epochs=EPOCHS,
    max_steps=10 if SMOKE else -1,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_steps=max(1, int(0.05 * total_steps)),
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=5 if SMOKE else 20,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=5 if SMOKE else 100,
    remove_unused_columns=False,
    report_to="none",
)

params = inspect.signature(TrainingArguments.__init__).parameters
if "eval_strategy" not in params:
    cfg["evaluation_strategy"] = cfg.pop("eval_strategy")
ignored = [k for k in cfg if k not in params]
if ignored:
    print("ignored by this transformers version:", ignored)
args = TrainingArguments(**{k: v for k, v in cfg.items() if k in params})

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100))

last = get_last_checkpoint(OUT_DIR) if os.path.isdir(OUT_DIR) else None
print("resuming from:", last)
trainer.train(resume_from_checkpoint=last)

In [ ]:
model.save_pretrained(SAVE_DIR)
tok.save_pretrained(SAVE_DIR)
print("saved to", SAVE_DIR)

In [ ]:
model.eval()
model.config.use_cache = True
for r in val_rows[:6]:
    ids = tok(r["prompt"], add_special_tokens=False, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=60, do_sample=False, pad_token_id=tok.pad_token_id)
    print("model:", tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip())
    print("gold :", r["completion"], "\n")